# 01 — Environment validation

Run this first on a fresh Panther Lake box. It checks that:

1. The bench Python dependencies import cleanly.
2. The OpenAI-compatible backend you brought up on `:9000` answers `/v1/models`.
3. We can locate the server PID and read its RSS (needed by notebook 02).
4. The iGPU memory probe can read `drm-resident-*` from the server's fdinfo (works on the `xe` driver — Panther Lake / Lunar Lake / Battlemage — and on `i915`; no root required).

**Operator prerequisite** — exactly one OpenAI-compatible backend is already listening on `127.0.0.1:9000` (OpenVINO Model Server, vLLM-XPU, llama.cpp `llama-server`, or any equivalent — see the README for example commands).

If `/v1/models` doesn't answer, fix that before running notebook 02 or 03.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from bench import client, memprobe
print('bench package imports OK')

In [ ]:
BASE_URL = 'http://127.0.0.1:9000/v1'

import openai
models = openai.OpenAI(base_url=BASE_URL, api_key='EMPTY').models.list()
for m in models.data:
    print(m.id)

In [ ]:
pid = memprobe.find_server_pid(9000)
print('server pid:', pid)
if pid is not None:
    rss_kb = memprobe._read_rss_kb(pid)
    print(f'baseline RSS: {rss_kb/1024:.1f} MB' if rss_kb else 'RSS unreadable (permissions?)')

In [ ]:
if pid is not None:
    igpu = memprobe._read_igpu_mem_mb(pid)
    if igpu is not None:
        print(f'server iGPU resident memory: {igpu:.1f} MB (from /proc/<pid>/fdinfo, xe driver)')
    else:
        print('server holds no /dev/dri fds yet — the iGPU column will populate once the backend issues GPU work in notebook 02')

In [ ]:
# Smoke-test one chat completion so notebook 02 doesn't fail on its first call.
result = client.call(
    base_url=BASE_URL,
    model=models.data[0].id,
    prompt='Reply with the single word: ready.',
    max_tokens=4,
    stream=False,
)
print(f'reply: {result.text!r}  ({result.total_latency_s:.2f}s, {result.completion_tokens} tok)')